In [ ]:
!apt-get update
!apt-get install golang-go
from google.colab import drive
drive.mount("/content/drive")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
golang-go is already the newest version (2:1.22~2build1).
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
Drive already mounted at /content/drive; to atte

**Implementación Concurrente (Worker Pool) con Prueba de Estrés**

En esta versión se implementa el patrón de concurrencia **Worker Pool** utilizando goroutines y primitivas de sincronización (sync.WaitGroup y sync.Mutex).

Para poder observar el beneficio real de la concurrencia y superar el *overhead* de creación de hilos en Go, se aplica una **Prueba de Estrés**. El dataset original de 1 035 observaciones válidas se multiplica por 1 000 en memoria RAM, generando una carga de procesamiento de **más de 1 millón de registros** (1 035 000 observaciones de 5 dimensiones).

In [ ]:
!go version
!uname -sr
!lscpu
!free -h

go version go1.22.2 linux/amd64
Linux 6.6.122+
Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.41
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma c

In [ ]:
%%writefile main_concurrente.go
package main

import (
	"encoding/csv"
	"fmt"
	"math"
	"math/rand"
	"os"
	"strconv"
	"sync"
	"time"
)

// Observacion representa una fila del dataset con sus 5 contaminantes.
type Observacion struct {
	Features []float64 // Vector multivariable: [PM10, PM2.5, SO2, NO2, CO]
	Cluster int   // Índice del centroide al que pertenece
}

// calcularDistancia aplica la fórmula de distancia euclidiana entre dos vectores.
func calcularDistancia(a, b []float64) float64 {
	sum := 0.0
	for i := range a {
		diff := a[i] - b[i]
		sum += diff * diff
	}
	return math.Sqrt(sum)
}

// kMeansConcurrente ejecuta el algoritmo de agrupamiento dividiendo la carga en múltiples hilos (Worker Pool).
func kMeansConcurrente(datos []Observacion, k int, maxIter int, numWorkers int) [][]float64 {
	numFeatures := len(datos[0].Features)
	centroides := make([][]float64, k)

	// 1. Inicialización: Seleccionar 'k' centroides iniciales al azar
	rand.Seed(time.Now().UnixNano())
	for i := 0; i < k; i++ {
		idx := rand.Intn(len(datos))
		centroides[i] = make([]float64, numFeatures)
		copy(centroides[i], datos[idx].Features)
	}

	// 2. Bucle principal de optimización
	for iter := 0; iter < maxIter; iter++ {
		cambiosTotales := 0
		var wg sync.WaitGroup
		var mu sync.Mutex // Protege la suma de los cambios totales para evitar Race Conditions

		// Calcular la cantidad de observaciones que procesará cada worker
		tamañoBloque := len(datos) / numWorkers

		// Fase A: Asignar cada observación al centroide más cercano (CONCURRENTE)
		for w := 0; w < numWorkers; w++ {
			inicio := w * tamañoBloque
			fin := inicio + tamañoBloque

			// El último worker procesa cualquier residuo de la división
			if w == numWorkers-1 {
				fin = len(datos)
			}

			wg.Add(1)

			// Lanzamiento de la Goroutine (Worker)
			go func(inicio, fin int) {
				defer wg.Done()
				cambiosLocal := 0 // Variable local para no bloquear la memoria en cada iteración

				for i := inicio; i < fin; i++ {
					minDist := math.MaxFloat64
					clusterAsignado := 0

					for j := 0; j < k; j++ {
						dist := calcularDistancia(datos[i].Features, centroides[j])
						if dist < minDist {
							minDist = dist
							clusterAsignado = j
						}
					}

					// Registrar si el punto cambió de grupo
					if datos[i].Cluster != clusterAsignado {
						datos[i].Cluster = clusterAsignado
						cambiosLocal++
					}
				}

				// Sección Crítica: Sumar los cambios del worker al contador global
				mu.Lock()
				cambiosTotales += cambiosLocal
				mu.Unlock()

			}(inicio, fin)
		}

		// Barrera: Esperar a que todos los workers terminen de asignar sus bloques
		wg.Wait()

		// Criterio de parada: Si ningún punto cambió de cluster, hemos terminado
		if cambiosTotales == 0 {
			fmt.Printf("-> Convergencia alcanzada en la iteración %d\n", iter)
			break
		}

		// Fase B: Recalcular la posición de los centroides (SECUENCIAL)
		// Se hace secuencial porque actualizar promedios compartidos generaría contención de memoria
		nuevosCentroides := make([][]float64, k)
		conteos := make([]int, k)
		for i := 0; i < k; i++ {
			nuevosCentroides[i] = make([]float64, numFeatures)
		}

		// Sumar los valores de todas las características por cluster
		for _, obs := range datos {
			c := obs.Cluster
			conteos[c]++
			for j := 0; j < numFeatures; j++ {
				nuevosCentroides[c][j] += obs.Features[j]
			}
		}

		// Dividir por la cantidad de puntos para obtener el nuevo centroide
		for i := 0; i < k; i++ {
			if conteos[i] > 0 {
				for j := 0; j < numFeatures; j++ {
					centroides[i][j] = nuevosCentroides[i][j] / float64(conteos[i])
				}
			}
		}
	}
	return centroides
}

func main() {
	rutaArchivo := "/content/drive/MyDrive/Concurrente/Procesado/dataset_kmeans_go.csv"

	file, err := os.Open(rutaArchivo)
	if err != nil {
		fmt.Printf("Error abriendo el archivo %s: %v\n", rutaArchivo, err)
		return
	}
	defer file.Close()

	reader := csv.NewReader(file)
	registros, err := reader.ReadAll()
	if err != nil {
		fmt.Println("Error leyendo el CSV:", err)
		return
	}

	var datasetBase []Observacion

	// Mapeo de columnas: 2:CO, 3:NO2, 4:PM10, 5:PM2.5, 6:SO2
	for i := 1; i < len(registros); i++ {
		valCO, _ := strconv.ParseFloat(registros[i][2], 64)
		valNO2, _ := strconv.ParseFloat(registros[i][3], 64)
		valPM10, _ := strconv.ParseFloat(registros[i][4], 64)
		valPM25, _ := strconv.ParseFloat(registros[i][5], 64)
		valSO2, _ := strconv.ParseFloat(registros[i][6], 64)

		obs := Observacion{
			Features: []float64{valPM10, valPM25, valSO2, valNO2, valCO},
			Cluster: -1,
		}
		datasetBase = append(datasetBase, obs)
	}

	// PRUEBA DE ESTRÉS: Multiplicador x1000 para evaluación
	var datasetMasivo []Observacion
	multiplicador := 1000
	for i := 0; i < multiplicador; i++ {
		datasetMasivo = append(datasetMasivo, datasetBase...)
	}

	fmt.Printf("Total de observaciones masivas procesadas: %d\n", len(datasetMasivo))

	// Configuración de hiperparámetros
	K := 3     // Cantidad de clusters a encontrar
	MaxIter := 100 // Límite de seguridad para evitar bucles infinitos
	NumWorkers := 4 // Cantidad de Goroutines dividiendo el trabajo

	// --- INICIO DE MEDICIÓN DE RENDIMIENTO ---
	inicio := time.Now()

	fmt.Printf("Ejecutando algoritmo K-Means Concurrente con %d Workers\n", NumWorkers)
	centroidesFinales := kMeansConcurrente(datasetMasivo, K, MaxIter, NumWorkers)

	duracion := time.Since(inicio)
	// --- FIN DE MEDICIÓN ---

	fmt.Printf("\n=== RESULTADOS CONCURRENTES ===\n")
	fmt.Printf("Tiempo de ejecución (T_concurrente): %v\n", duracion)
	fmt.Println("Centroides finales [PM10, PM2.5, SO2, NO2, CO]:")
	for i, c := range centroidesFinales {
		fmt.Printf(" Cluster %d: %.4f\n", i, c)
	}
}

Overwriting main_concurrente.go


In [ ]:
!go run main_concurrente.go

Total de observaciones masivas procesadas: 1035000
Ejecutando algoritmo K-Means Concurrente con 4 Workers
-> Convergencia alcanzada en la iteración 17

=== RESULTADOS CONCURRENTES ===
Tiempo de ejecución (T_concurrente): 837.210526ms
Centroides finales [PM10, PM2.5, SO2, NO2, CO]:
 Cluster 0: [19.6568 12.5415 14.8296 6.2732 298.2777]
 Cluster 1: [18.3203 11.5569 13.5596 6.5114 358.4277]
 Cluster 2: [18.9841 12.3515 12.7142 6.2845 246.5373]


In [ ]:
!go build -o kmeans_concurrente main_concurrente.go

In [ ]:
!sudo apt-get install -y time

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  time
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 45.2 kB of archives.
After this operation, 129 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 time amd64 1.9-0.2build1 [45.2 kB]
Fetched 45.2 kB in 0s (105 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package time.
(Reading database ... 137127 files and directories currently 

In [ ]:
%%bash
for i in 1 2 3 4 5
do

    echo "PRUEBA $i - CONCURRENTE"
    /usr/bin/time -f 'Tiempo total: %e s | CPU: %P | RAM máxima: %M KiB' ./kmeans_concurrente
done

PRUEBA 1 - CONCURRENTE
Total de observaciones masivas procesadas: 1035000
Ejecutando algoritmo K-Means Concurrente con 4 Workers
-> Convergencia alcanzada en la iteración 24

=== RESULTADOS CONCURRENTES ===
Tiempo de ejecución (T_concurrente): 1.143583425s
Centroides finales [PM10, PM2.5, SO2, NO2, CO]:
 Cluster 0: [19.1172 12.4054 13.0164 6.2894 248.0573]
 Cluster 1: [19.4064 12.3933 14.4453 6.2424 301.5578]
 Cluster 2: [18.4700 11.6429 13.8242 6.5876 361.8309]
PRUEBA 2 - CONCURRENTE
Total de observaciones masivas procesadas: 1035000
Ejecutando algoritmo K-Means Concurrente con 4 Workers
-> Convergencia alcanzada en la iteración 15

=== RESULTADOS CONCURRENTES ===
Tiempo de ejecución (T_concurrente): 1.378172611s
Centroides finales [PM10, PM2.5, SO2, NO2, CO]:
 Cluster 0: [18.9841 12.3515 12.7142 6.2845 246.5373]
 Cluster 1: [18.3203 11.5569 13.5596 6.5114 358.4277]
 Cluster 2: [19.6568 12.5415 14.8296 6.2732 298.2777]
PRUEBA 3 - CONCURRENTE
Total de observaciones masivas procesadas: 

Tiempo total: 1.56 s | CPU: 167% | RAM máxima: 98652 KiB
Tiempo total: 1.85 s | CPU: 104% | RAM máxima: 134004 KiB
Tiempo total: 2.54 s | CPU: 94% | RAM máxima: 122648 KiB
Tiempo total: 1.66 s | CPU: 138% | RAM máxima: 96312 KiB
Tiempo total: 1.29 s | CPU: 168% | RAM máxima: 115428 KiB
